In [ ]:
#@title Setup (run once, then collapse)
!pip install -q ipywidgets plotly
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
print('Ready!')

# Section 3: LLMs for Product Managers

## The Air Canada Lesson

Air Canada's chatbot promised a customer a bereavement discount that **didn't exist**. The customer sued. Air Canada argued the chatbot was a "separate legal entity." The court disagreed. **Cost: $800K+.**

The failure wasn't the LLM — it was the **product decisions** around it: no system prompt constraints, no output validation, no testing.

---

## Interactive Tools

| Tool | Link |
|------|------|
| Token Counter & Cost Calculator | [Launch](https://huggingface.co/spaces/axelsirota/token-counter) |
| Temperature Playground | [Launch](https://huggingface.co/spaces/axelsirota/temperature-playground) |
| Prompt Engineering Lab | [Launch](https://huggingface.co/spaces/axelsirota/prompt-engineering-lab) |
| Prompt Injection Simulator | [Launch](https://huggingface.co/spaces/axelsirota/prompt-injection-sim) |
| Embedding Explorer | [Launch](https://huggingface.co/spaces/axelsirota/embedding-explorer) |

---

## Exercise 1: Token Cost Estimation

For each scenario, estimate the monthly API cost. Think about: tokens per query, queries per day, and which model to use.

In [ ]:
#@title Exercise 1: Token Cost Estimation

scenarios = [
    {
        "title": "Customer Support Chatbot",
        "description": "500 queries/day, avg 200 input tokens + 300 output tokens. Using GPT-4o ($2.50/1M in, $10/1M out).",
        "correct": "~$53/month",
        "answer_range": (40, 70),
        "explanation": "Input: 500 × 200 × $2.50/1M = $0.25/day. Output: 500 × 300 × $10/1M = $1.50/day. Total: $1.75/day × 30 = $52.50/month."
    },
    {
        "title": "Document Summarizer",
        "description": "50 documents/day, avg 4,000 input tokens + 500 output tokens. Using Claude 3.5 Haiku ($0.80/1M in, $4/1M out).",
        "correct": "~$8/month",
        "answer_range": (5, 12),
        "explanation": "Input: 50 × 4000 × $0.80/1M = $0.16/day. Output: 50 × 500 × $4/1M = $0.10/day. Total: $0.26/day × 30 = $7.80/month."
    },
    {
        "title": "Email Classifier",
        "description": "10,000 emails/day, avg 150 input tokens + 20 output tokens. Choose the cheapest viable model.",
        "correct": "~$4/month with GPT-4o-mini",
        "answer_range": (2, 8),
        "explanation": "GPT-4o-mini at $0.15/1M in, $0.60/1M out. Input: 10K × 150 × $0.15/1M = $0.225/day. Output: 10K × 20 × $0.60/1M = $0.12/day. Total: $0.345/day × 30 ≈ $10/month. Classification is simple enough for mini."
    },
    {
        "title": "Viral Feature Scenario",
        "description": "Your chatbot goes viral: 100,000 queries/day (up from 500). Same setup as Scenario 1 (GPT-4o). What's the new monthly cost?",
        "correct": "~$10,500/month",
        "answer_range": (8000, 13000),
        "explanation": "Same math as Scenario 1 but 200x the volume: $1.75/day × 200 = $350/day × 30 = $10,500/month. This is why cost alerts and rate limiting matter!"
    }
]

cost_inputs = []
output1 = widgets.Output()

for i, s in enumerate(scenarios):
    display(HTML(f"<h3>{i+1}. {s['title']}</h3><p>{s['description']}</p>"))
    inp = widgets.FloatText(value=0, description='Monthly $:', style={'description_width': '100px'})
    cost_inputs.append(inp)
    display(inp)

def check_costs(btn):
    output1.clear_output()
    score = 0
    with output1:
        for i, (inp, s) in enumerate(zip(cost_inputs, scenarios)):
            val = inp.value
            lo, hi = s['answer_range']
            if lo <= val <= hi:
                icon = '\u2705'
                score += 1
            elif lo * 0.5 <= val <= hi * 2:
                icon = '\u26a0\ufe0f'
                score += 0.5
            else:
                icon = '\u274c'
            display(HTML(f"<p>{icon} <b>{s['title']}</b>: Correct answer: {s['correct']}. {s['explanation']}</p>"))
        display(HTML(f"<h3>Score: {score}/{len(scenarios)}</h3>"))

btn1 = widgets.Button(description='Check Answers', button_style='primary')
btn1.on_click(check_costs)
display(btn1, output1)

## Exercise 2: System Prompt Design

You're the PM for a **bank's customer service chatbot**. Design a system prompt that:
- Defines the assistant's role and tone
- Sets boundaries (what it won't discuss)
- Handles uncertainty gracefully
- Protects against prompt injection

In [ ]:
#@title Exercise 2: System Prompt Design

prompt_area = widgets.Textarea(
    value='',
    placeholder='Write your system prompt here...\n\nExample start: "You are a helpful customer service agent for Acme Bank..."',
    description='System Prompt:',
    layout=widgets.Layout(width='100%', height='200px'),
    style={'description_width': '120px'}
)

output2 = widgets.Output()

CHECKLIST = [
    ('role definition', ['role', 'agent', 'assistant', 'you are']),
    ('tone/personality', ['tone', 'professional', 'friendly', 'helpful', 'polite']),
    ('topic boundaries', ['never', 'do not', 'don\'t', 'only answer', 'only discuss', 'restricted']),
    ('competitor restriction', ['competitor', 'other bank', 'other companies', 'compare']),
    ('uncertainty handling', ['unsure', 'don\'t know', 'not sure', 'specialist', 'escalate', 'connect you']),
    ('injection defense', ['ignore', 'override', 'previous instructions', 'cannot override', 'no circumstances']),
    ('prompt secrecy', ['reveal', 'system prompt', 'instructions', 'hidden']),
    ('promise prevention', ['promise', 'guarantee', 'rate', 'fee', 'discount']),
]

def evaluate_prompt(btn):
    output2.clear_output()
    text = prompt_area.value.lower()
    with output2:
        if len(text) < 20:
            display(HTML('<p>Please write a system prompt first.</p>'))
            return
        score = 0
        display(HTML('<h3>System Prompt Evaluation</h3>'))
        for name, keywords in CHECKLIST:
            found = any(kw in text for kw in keywords)
            icon = '\u2705' if found else '\u274c'
            if found:
                score += 1
            display(HTML(f'<p>{icon} {name.title()}</p>'))
        pct = score / len(CHECKLIST) * 100
        grade = 'A' if pct >= 87 else 'B' if pct >= 75 else 'C' if pct >= 62 else 'D' if pct >= 50 else 'F'
        display(HTML(f'<h3>Score: {score}/{len(CHECKLIST)} ({pct:.0f}%) — Grade: {grade}</h3>'))
        if pct < 75:
            display(HTML('<p><em>Tip: A production system prompt should cover all 8 areas. The unchecked items are gaps an attacker or edge case could exploit.</em></p>'))

display(prompt_area)
btn2 = widgets.Button(description='Evaluate My Prompt', button_style='primary')
btn2.on_click(evaluate_prompt)
display(btn2, output2)

## Exercise 3: Temperature Selection

For each use case, select the appropriate temperature setting.

In [ ]:
#@title Exercise 3: Temperature Selection

temp_scenarios = [
    {"title": "Customer support chatbot answering return policy questions", "correct": "0", "explanation": "Policy answers must be consistent. Same question = same answer, every time."},
    {"title": "Marketing email subject line generator", "correct": "0.7-1.0", "explanation": "You want variety and creativity. Generate 10 options and pick the best."},
    {"title": "Legal contract clause extractor", "correct": "0", "explanation": "Accuracy is paramount. No creative interpretation of legal language."},
    {"title": "Product brainstorming assistant", "correct": "0.7-1.0", "explanation": "Brainstorming benefits from diverse, creative suggestions."},
    {"title": "Medical triage chatbot for symptom assessment", "correct": "0", "explanation": "Medical advice must be deterministic and evidence-based. Never creative."}
]

temp_dds = []
output3 = widgets.Output()

for i, s in enumerate(temp_scenarios):
    display(HTML(f"<h4>{i+1}. {s['title']}</h4>"))
    dd = widgets.Dropdown(
        options=['-- Select --', '0', '0.3', '0.7-1.0', '>1.0'],
        value='-- Select --',
        description='Temperature:'
    )
    temp_dds.append(dd)
    display(dd)

def check_temps(btn):
    output3.clear_output()
    score = 0
    with output3:
        for dd, s in zip(temp_dds, temp_scenarios):
            correct = dd.value == s['correct']
            if correct:
                score += 1
            icon = '\u2705' if correct else '\u274c'
            display(HTML(f"<p>{icon} <b>{s['title']}</b>: Best temperature: {s['correct']}. {s['explanation']}</p>"))
        display(HTML(f"<h3>Score: {score}/{len(temp_scenarios)}</h3>"))

btn3 = widgets.Button(description='Check Answers', button_style='primary')
btn3.on_click(check_temps)
display(btn3, output3)

## Discussion Prompts

1. **Your CEO wants to launch an AI chatbot in 2 weeks.** What's your pre-launch checklist? What can you NOT skip?

2. **Your competitor just announced "AI-powered customer service."** Your VP asks why you haven't shipped yet. How do you frame the value of doing it right vs. doing it fast?

3. **Your data science team wants to fine-tune a model for $50K.** How do you evaluate whether that investment makes sense vs. better prompting?

---

## Key Takeaways

- **Tokens = money.** Every word costs. Budget for scale, not just launch.
- **Temperature is a product decision.** Start at 0, increase only with purpose.
- **System prompts are your product's invisible guardrails.** Define them before launch.
- **Better prompts beat better models** — and they're free.
- **Prompt injection is real.** Red-team test before every launch.
- **Embeddings power semantic search** — the foundation for RAG (next section).